# Step 4+5: baseline inference, prompts v1/v2/v3 (Exp 1, 2, 3)

Loads base Qwen2.5-Coder-1.5B-Instruct **once** and runs it on all 1,034 dev prompts for each of v1, v2, v3 in turn.

- Model is public: no HF token needed.
- Tries vLLM first (fast, single T4, dtype float16). If vLLM fails to install or load, falls back to batched Hugging Face `generate` automatically in this same notebook — no manual intervention needed.
- Greedy decoding (temperature 0), `max_new_tokens=256`, fixed seed.
- Input: `/kaggle/input/<your-dataset-slug>/prompts_v1_dev.jsonl`, `prompts_v2_dev.jsonl`, `prompts_v3_dev.jsonl` (each row: idx, db_id, question, prompt — no gold SQL).
- Output: `/kaggle/working/predictions_v1.jsonl`, `predictions_v2.jsonl`, `predictions_v3.jsonl`, and `run_metadata.json`.

**Settings before Run All:** Settings → Accelerator: GPU T4 x2, Internet: On. (This notebook only uses one of the two T4s — that's intentional, see project notes.) Add Input → your uploaded dataset. No secrets needed.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single T4 on purpose

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SEED = 0
MAX_NEW_TOKENS = 256
MAX_MODEL_LEN = 4096  # covers our longest v3 prompt (~3528 tok) + generation headroom
CHUNK_SIZE = 32        # prompts per timed batch, for per-example latency bookkeeping

INPUT_DIR = "/kaggle/input"
# auto-detect the uploaded dataset folder (whatever slug Kaggle assigned it)
dataset_dirs = [os.path.join(INPUT_DIR, d) for d in os.listdir(INPUT_DIR)]
DATA_DIR = next(d for d in dataset_dirs if os.path.exists(os.path.join(d, "prompts_v1_dev.jsonl")))
print("reading prompts from:", DATA_DIR)

OUT_DIR = "/kaggle/working"
VERSIONS = ["v1", "v2", "v3"]

In [ ]:
import json

def load_prompts(version):
    path = os.path.join(DATA_DIR, f"prompts_{version}_dev.jsonl")
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
# try vLLM first; fall back to batched HF generate on any failure (install, import, or load)
import subprocess, sys, traceback

USE_VLLM = False
backend_error = None
try:
    rc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"]).returncode
    if rc != 0:
        raise RuntimeError(f"pip install vllm exited {rc}")
    from vllm import LLM, SamplingParams
    llm = LLM(model=MODEL_NAME, dtype="float16", gpu_memory_utilization=0.85,
              max_model_len=MAX_MODEL_LEN, seed=SEED)
    sampling_params = SamplingParams(temperature=0, max_tokens=MAX_NEW_TOKENS, seed=SEED)
    USE_VLLM = True
    print("vLLM loaded OK, using vLLM backend")
except Exception as e:
    backend_error = f"{type(e).__name__}: {e}"
    print("vLLM path failed, falling back to batched HF generate.\n", traceback.format_exc())
    USE_VLLM = False

In [ ]:
if not USE_VLLM:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    torch.manual_seed(SEED)
    hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if hf_tokenizer.pad_token is None:
        hf_tokenizer.pad_token = hf_tokenizer.eos_token
    hf_tokenizer.padding_side = "left"  # required for correct batched causal-LM generation
    hf_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to("cuda")
    hf_model.eval()
    HF_BATCH_SIZE = 8  # conservative for a 16GB T4 with up to ~3800-token prompts


def generate_chunk(prompts):
    """Returns a list of raw completion strings, same order as `prompts`."""
    if USE_VLLM:
        outputs = llm.generate(prompts, sampling_params)
        return [o.outputs[0].text for o in outputs]
    texts = []
    for i in range(0, len(prompts), HF_BATCH_SIZE):
        batch = prompts[i:i + HF_BATCH_SIZE]
        enc = hf_tokenizer(batch, return_tensors="pt", padding=True, truncation=True,
                            max_length=MAX_MODEL_LEN).to("cuda")
        with torch.no_grad():
            out = hf_model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                     pad_token_id=hf_tokenizer.pad_token_id)
        gen_only = out[:, enc["input_ids"].shape[1]:]
        texts.extend(hf_tokenizer.batch_decode(gen_only, skip_special_tokens=True))
    return texts

In [ ]:
import time

run_metadata = {
    "model": MODEL_NAME, "backend": "vllm" if USE_VLLM else "transformers",
    "backend_fallback_reason": backend_error, "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS, "decoding": "greedy",
    "versions": {},
}

try:
    import torch as _torch
    run_metadata["gpu_name"] = _torch.cuda.get_device_name(0)
except Exception:
    run_metadata["gpu_name"] = "unknown"

for pkg in ["torch", "transformers"] + (["vllm"] if USE_VLLM else []):
    try:
        mod = __import__(pkg)
        run_metadata[f"{pkg}_version"] = getattr(mod, "__version__", "unknown")
    except Exception:
        run_metadata[f"{pkg}_version"] = "not installed"

for version in VERSIONS:
    rows = load_prompts(version)
    out_path = os.path.join(OUT_DIR, f"predictions_{version}.jsonl")
    version_start = time.time()
    with open(out_path, "w", encoding="utf-8") as f:
        for i in range(0, len(rows), CHUNK_SIZE):
            chunk = rows[i:i + CHUNK_SIZE]
            prompts = [r["prompt"] for r in chunk]
            t0 = time.time()
            raw_outputs = generate_chunk(prompts)
            elapsed = time.time() - t0
            per_example_time = elapsed / len(chunk)
            for r, raw in zip(chunk, raw_outputs):
                f.write(json.dumps({
                    "idx": r["idx"], "db_id": r["db_id"], "question": r["question"],
                    "raw_output": raw, "gen_time_sec": per_example_time,
                }) + "\n")
            print(f"{version}: {i + len(chunk)}/{len(rows)} done, last chunk {elapsed:.1f}s")
    version_elapsed = time.time() - version_start
    run_metadata["versions"][version] = {"n_examples": len(rows), "total_time_sec": version_elapsed}
    print(f"=== {version} done in {version_elapsed:.1f}s, wrote {out_path} ===")

with open(os.path.join(OUT_DIR, "run_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)
print(json.dumps(run_metadata, indent=2))

## Download these files from /kaggle/working/ when done:
- `predictions_v1.jsonl`
- `predictions_v2.jsonl`
- `predictions_v3.jsonl`
- `run_metadata.json`